# Ball detection on a Kaggle GPU

Stage 1 of the ball tracker (`tracking/ball_tracking.py --detect-only`) runs the
YOLOv8 detector on full-resolution tiles of every frame. It is too heavy for most
laptops, so it runs here on a free GPU; the result is a small **candidates CSV**.
Stage 2 (the tracker) is cheap and runs anywhere: `pipeline.py` picks the CSV up
automatically.

Unlike Colab, a saved Kaggle version runs in the background: once it has started
you can close the browser and turn the computer off.

**Setup**
1. *Input → Add Input → Upload*: create a **private** dataset with `ball_tracker.pt`
   (from the GitHub Release) and your videos. Keep the file names: a clip's results
   are named after its video.
2. *Settings*: set *Accelerator* to a GPU (T4) and turn *Internet* on. If the Internet
   option is missing, verify your phone number on your Kaggle account.
3. *Save Version → Save & Run All (Commit)*.

## 1. Check the GPU
If `CUDA available: False`, set *Settings → Accelerator* to a GPU and run again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Get the code
Clones the repository, so this notebook always runs the same tracker as the project.

In [ ]:
!git clone -q https://github.com/marcomassa-unitn/tennis_players_tracking.git
%cd tennis_players_tracking
!pip -q install ultralytics

## 3. Find the weights and the videos
They are searched for in all the inputs, so the name of the dataset does not matter.

In [ ]:
import glob
files = glob.glob('/kaggle/input/**/*', recursive=True)
MODELS = [f for f in files if f.endswith('ball_tracker.pt')]
VIDEOS = sorted(f for f in files if f.endswith('.mp4'))

assert MODELS, 'ball_tracker.pt not found: add it to the dataset attached to this notebook'
assert VIDEOS, 'No .mp4 video found: add the videos to the dataset attached to this notebook'
MODEL = MODELS[0]
print('Model:', MODEL)
print('Videos:', *VIDEOS, sep='\n  ')

## 4. Detect the ball candidates (GPU)
About 8 tiles per 1080p frame; a 1000-frame clip takes a few minutes on a T4.

In [ ]:
for v in VIDEOS:
    !python tracking/ball_tracking.py --video "{v}" --model "{MODEL}" --detect-only

## 5. Save the candidates
Only `/kaggle/working` is kept, so the CSVs go there and the code is removed.

When the run is over, open the saved version and download the CSVs from its *Output*
section. On your PC, put each one in `outputs/<clip>/ball_coordinates/` and run
`python pipeline.py --video data/<clip>.mp4`.

In [ ]:
import shutil
candidates = glob.glob('outputs/*/ball_coordinates/ball_candidates_*.csv')
assert len(candidates) == len(VIDEOS), f'{len(candidates)} CSVs for {len(VIDEOS)} videos'
for csv_path in candidates:
    print('Saved', shutil.copy(csv_path, '/kaggle/working/'))
%cd /kaggle/working
!rm -rf tennis_players_tracking